# HuggingFace Model Helpers

Orchestration functions (run locally, SSH to Isambard) for pre-downloading
HuggingFace models, plus compute-node functions for loading models in SBATCH
jobs. Compute nodes have no internet, so models must be pre-cached on login nodes.

In [ ]:
#|default_exp models

In [ ]:
#|export
import os
from dataclasses import dataclass, field
from functools import partial
from typing import Any

from isambard_utils.config import IsambardConfig
from isambard_utils.ssh import run as ssh_run, arun as async_ssh_run, _get_config, _run_sync

_fprint = partial(print, flush=True)

## Orchestration (run locally, SSH to Isambard)

In [ ]:
#|export
def _shlex_quote(s: str) -> str:
    import shlex
    return shlex.quote(s)

In [ ]:
#|export
def _hf_cache_dir(config: IsambardConfig) -> str:
    """Return the HF cache directory on the remote."""
    return config.hf_cache_dir

In [ ]:
#|export
async def _aremote_python(script: str, *, config: IsambardConfig, timeout: int = 600) -> str:
    """Run a Python script in the remote venv and return stdout (async)."""
    import base64, subprocess
    b64 = base64.b64encode(script.encode()).decode()
    cmd = (
        f"cd {config.project_dir} && source .venv/bin/activate && "
        f"echo {b64} | base64 -d | python"
    )
    try:
        result = await async_ssh_run(f"bash -lc {_shlex_quote(cmd)}", config=config, timeout=timeout)
    except subprocess.CalledProcessError as e:
        raise RuntimeError(
            f"Remote python script failed (exit {e.returncode}).\n"
            f"--- script ---\n{script}\n"
            f"--- stderr ---\n{e.stderr or '(empty)'}"
        ) from e
    return result.stdout

In [ ]:
#|export
def _remote_python(script: str, *, config: IsambardConfig, timeout: int = 600) -> str:
    """Run a Python script in the remote venv and return stdout."""
    return _run_sync(_aremote_python(script, config=config, timeout=timeout))

In [ ]:
#|export
async def acheck_model(model_name: str, *, config: IsambardConfig | None = None) -> bool:
    """Check if a HuggingFace model is already cached on Isambard (async).

    Verifies the snapshot directory exists AND contains a config.json
    (not just metadata like README/LICENSE from partial downloads).

    Args:
        model_name: HuggingFace model ID.
        config: Isambard configuration.
    """
    config = _get_config(config)
    cache_dir = _hf_cache_dir(config)
    # HF hub stores snapshots under models--org--name
    model_dir = model_name.replace("/", "--")
    path = f"{cache_dir}/models--{model_dir}"
    # Check that the snapshot directory exists AND has a config.json
    # (incomplete downloads may only have README.md / LICENSE)
    check_cmd = (
        f"test -d {_shlex_quote(path)} && "
        f"ls {_shlex_quote(path)}/snapshots/*/config.json >/dev/null 2>&1"
    )
    result = await async_ssh_run(check_cmd, config=config, check=False)
    return result.returncode == 0

In [ ]:
#|export
def check_model(model_name: str, *, config: IsambardConfig | None = None) -> bool:
    """Check if a HuggingFace model is already cached on Isambard.

    Verifies the snapshot directory exists AND contains a config.json
    (not just metadata like README/LICENSE from partial downloads).

    Args:
        model_name: HuggingFace model ID.
        config: Isambard configuration.
    """
    return _run_sync(acheck_model(model_name, config=config))

In [ ]:
#|export
async def _aprecache_dynamic_modules(model_name: str, *, cache_dir: str,
                                     config: IsambardConfig,
                                     print_fn=_fprint) -> None:
    """Pre-cache transformers dynamic modules for trust_remote_code models.

    Models with ``auto_map`` in their config.json use custom Python files that
    transformers resolves via ``dynamic_module_utils``. These are stored in a
    separate modules cache (``~/.cache/huggingface/modules/``) that must be
    populated on the login node (which has internet) so that offline compute
    nodes can load them.

    This is a no-op for models without ``auto_map``.
    """
    # HF_HOME must point to a shared Lustre path so that dynamic modules
    # are cached where compute nodes can access them (not ~/.cache/huggingface/).
    hf_home = f"{config.project_dir}/.hf_home"
    script = "\n".join([
        "import json, glob, os",
        f"os.environ['HF_HOME'] = {hf_home!r}",
        f"os.environ['HF_HUB_CACHE'] = {cache_dir!r}",
        f"model_dir = {model_name!r}.replace('/', '--')",
        f"pattern = os.path.join({cache_dir!r}, f'models--{{model_dir}}', 'snapshots', '*', 'config.json')",
        "cfgs = glob.glob(pattern)",
        "if cfgs:",
        "    cfg = json.load(open(cfgs[0]))",
        "    auto_map = cfg.get('auto_map')",
        "    if auto_map:",
        # Resolve ALL auto_map entries (config, model, tokenizer, etc.)
        # AutoConfig alone only caches configuration.py; we also need modeling.py etc.
        "        from transformers.dynamic_module_utils import get_class_from_dynamic_module",
        "        for class_ref in auto_map.values():",
        f"            get_class_from_dynamic_module(class_ref, {model_name!r}, cache_dir={cache_dir!r})",
        "        print('pre-cached dynamic modules')",
        "    else:",
        "        print('no auto_map')",
        "else:",
        "    print('no config.json found')",
    ])
    stdout = await _aremote_python(script, config=config, timeout=600)
    result = stdout.strip().split("\n")[-1]
    print_fn(f"aensure_model: {model_name}: {result}")

In [ ]:
#|export
async def aensure_model(model_name: str, *, config: IsambardConfig | None = None,
                        token: str | None = None, timeout: int = 7200,
                        print_fn=_fprint) -> str:
    """Pre-download a HuggingFace model to the Isambard cache via the login node (async).

    Uses huggingface_hub.snapshot_download() in the remote venv. Returns the
    remote snapshot path. Skips download if model is already cached.

    After downloading, also pre-caches any transformers dynamic modules
    (for models with ``auto_map`` in config.json / trust_remote_code) so
    that offline compute nodes can load them.

    Args:
        model_name: HuggingFace model ID (e.g. "BAAI/bge-large-en-v1.5").
        config: Isambard configuration.
        token: Optional HuggingFace token for gated models.
        timeout: SSH timeout in seconds (default 2 hours for large models).
        print_fn: Callable for progress logging (default: print).
    """
    config = _get_config(config)
    cache_dir = _hf_cache_dir(config)
    if token is None:
        token = os.environ.get("HF_TOKEN")

    # Skip download if already cached (avoids 401 for gated models)
    if await acheck_model(model_name, config=config):
        print_fn(f"aensure_model: {model_name}: already cached")
        model_dir = model_name.replace("/", "--")
        # Return the snapshot path by reading the refs/main pointer
        script = (
            f"import os; "
            f"refs = os.path.join({cache_dir!r}, 'models--{model_dir}', 'refs', 'main'); "
            f"rev = open(refs).read().strip() if os.path.exists(refs) else 'unknown'; "
            f"snap = os.path.join({cache_dir!r}, 'models--{model_dir}', 'snapshots', rev); "
            f"print(snap)"
        )
        stdout = await _aremote_python(script, config=config, timeout=30)
        snapshot_path = stdout.strip().split("\n")[-1]
    else:
        print_fn(f"aensure_model: {model_name}: downloading...")
        token_arg = f", token={token!r}" if token else ""
        script = (
            f"from huggingface_hub import snapshot_download; "
            f"path = snapshot_download({model_name!r}, cache_dir={cache_dir!r}{token_arg}); "
            f"print(path)"
        )
        stdout = await _aremote_python(script, config=config, timeout=timeout)
        snapshot_path = stdout.strip().split("\n")[-1]
        print_fn(f"aensure_model: {model_name}: downloaded to {snapshot_path}")

    # Pre-cache transformers dynamic modules for trust_remote_code models
    await _aprecache_dynamic_modules(model_name, cache_dir=cache_dir, config=config, print_fn=print_fn)

    return snapshot_path

In [ ]:
#|export
def ensure_model(model_name: str, *, config: IsambardConfig | None = None,
                 token: str | None = None, timeout: int = 7200,
                 print_fn=_fprint) -> str:
    """Pre-download a HuggingFace model to the Isambard cache via the login node.

    Uses huggingface_hub.snapshot_download() in the remote venv. Returns the
    remote snapshot path. Skips download if model is already cached.

    After downloading, also pre-caches any transformers dynamic modules
    (for models with ``auto_map`` in config.json / trust_remote_code) so
    that offline compute nodes can load them.

    Args:
        model_name: HuggingFace model ID (e.g. "BAAI/bge-large-en-v1.5").
        config: Isambard configuration.
        token: Optional HuggingFace token for gated models.
        timeout: SSH timeout in seconds (default 2 hours for large models).
        print_fn: Callable for progress logging (default: print).
    """
    return _run_sync(aensure_model(model_name, config=config, token=token, timeout=timeout, print_fn=print_fn))

## Compute-node functions (run on Isambard in SBATCH jobs)

These classes and functions are defined in `llm_runner.models` and
re-exported here for backward compatibility.

In [ ]:
#|export
from llm_runner.models import (
    set_model_env, _resolve_dtype,
    EmbeddingModel, load_embedding_model,
    LLM, VllmLLM, _load_vllm, load_llm,
)